# ML量化框架使用示例

本Notebook展示如何使用可插拔的机器学习模型进行量化交易策略开发。

## 1. 导入必要的模块

In [1]:
import sys
sys.path.insert(0, '..')

# 数据加载与特征工程
from ml_framework.data_loader import StockDataLoader, time_series_split
from ml_framework.feature_engineering import FeatureEngineer
from ml_framework.backtester import Backtester
from ml_framework.config import DATA_PATH

# 模型（可插拔）
from ml_framework.models.sklearn_models import (
    RidgeRegressionModel, 
    RandomForestModel, 
    XGBoostModel, 
    LightGBMModel
)
from ml_framework.models.pytorch_models import MLPModel, LSTMModel

## 2. 数据加载

In [2]:
loader = StockDataLoader(DATA_PATH)
df = loader.load(years_back=10)

# 选择50只样本股票
selected_codes = loader.select_sample_codes(n=50)
df = df[df['code'].isin(selected_codes)]

print(f"股票数量: {df['code'].nunique()}")
print(f"日期范围: {df['date'].min()} ~ {df['date'].max()}")

df.head()

📂 加载数据: /Users/harry/quant-learning/ml_framework/../data/a_stock_history_price.csv
   原始: 7,783,131 行
   加载后: 6,706,561 行 | 6403 只股票
股票数量: 50
日期范围: 2021-02-18 00:00:00 ~ 2026-02-13 00:00:00


,date,code,name,open,close,high,low,price_change,pct_change,volume,turnover_rate,market_cap,dividends,stock_splits,amount
37634,2021-02-18,000049,德赛电池,45.00,44.71,45.68,42.97,0.15,0.34,6674868.0,1.74,1.719719e+10,0.0,0.0,2.984333e+08
37635,2021-02-19,000049,德赛电池,44.48,43.72,45.61,43.11,-0.99,-2.21,5812282.0,1.51,1.681640e+10,0.0,0.0,2.541130e+08
37636,2021-02-22,000049,德赛电池,43.60,41.16,43.65,41.16,-2.56,-5.86,8508151.0,2.21,1.583172e+10,0.0,0.0,3.501955e+08
37637,2021-02-23,000049,德赛电池,40.19,40.84,41.63,40.02,-0.32,-0.78,5122607.0,1.33,1.570864e+10,0.0,0.0,2.092073e+08
37638,2021-02-24,000049,德赛电池,41.21,39.62,41.52,39.41,-1.22,-2.99,6023507.0,1.57,1.523938e+10,0.0,0.0,2.386513e+08


## 3. 数据集划分（时序划分）

In [3]:
train_df, val_df, test_df = time_series_split(df)

print(f"训练集: {len(train_df)} 行")
print(f"验证集: {len(val_df)} 行")
print(f"测试集: {len(test_df)} 行")


⏰ 时间序列数据集划分...
   训练集: 27,447 (2021-02-18 ~ 2024-02-19)
   验证集: 11,201 (2024-02-20 ~ 2025-02-20)
   测试集: 11,855 (2025-02-21 ~ 2026-02-13)
训练集: 27447 行
验证集: 11201 行
测试集: 11855 行


## 4. 特征工程

In [4]:
from sklearn.preprocessing import StandardScaler

fe = FeatureEngineer()

# 生成特征
train_features = fe.create_features(train_df)
val_features = fe.create_features(val_df)
test_features = fe.create_features(test_df)

# 标准化
scaler = StandardScaler()
X_train, y_train, _ = fe.prepare_xy(train_features, scaler, fit_scaler=True)
X_val, y_val, _ = fe.prepare_xy(val_features, scaler, fit_scaler=False)
X_test, y_test, _ = fe.prepare_xy(test_features, scaler, fit_scaler=False)

feature_cols = fe.feature_cols
print(f"特征数量: {len(feature_cols)}")
print(f"特征列表: {feature_cols[:5]} ...")


🔧 构建特征...
   特征数: 40
   有效样本: 24,319

🔧 构建特征...
   特征数: 40
   有效样本: 7,989

🔧 构建特征...
   特征数: 40
   有效样本: 8,398


TypeError: object of type 'StandardScaler' has no len()

## 5. 模型训练与评估

这里演示**可插拔**的特性：只需更换模型类，其他代码完全相同！

### 5.1 随机森林

In [ ]:
# 创建模型
model_rf = RandomForestModel(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# 训练
history_rf = model_rf.fit(X_train, y_train, X_val, y_val)

# 评估
test_metrics_rf = model_rf.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_rf.items():
    print(f"   {k}: {v:.4f}")

### 5.2 XGBoost

In [ ]:
# 更换模型，其他代码不变！
model_xgb = XGBoostModel(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1
)

history_xgb = model_xgb.fit(X_train, y_train, X_val, y_val)

test_metrics_xgb = model_xgb.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_xgb.items():
    print(f"   {k}: {v:.4f}")

### 5.3 LightGBM

In [ ]:
model_lgb = LightGBMModel(
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.05
)

history_lgb = model_lgb.fit(X_train, y_train, X_val, y_val)

test_metrics_lgb = model_lgb.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_lgb.items():
    print(f"   {k}: {v:.4f}")

### 5.4 MLP神经网络（PyTorch）

In [ ]:
# 深度学习模型同样可以无缝替换
model_mlp = MLPModel(
    input_dim=X_train.shape[1],
    hidden_dims=[128, 64, 32],
    dropout_rate=0.3,
    epochs=50,
    batch_size=256,
    lr=0.001
)

history_mlp = model_mlp.fit(X_train, y_train, X_val, y_val)

test_metrics_mlp = model_mlp.evaluate(X_test, y_test)
print(f"\n测试集表现:")
for k, v in test_metrics_mlp.items():
    print(f"   {k}: {v:.4f}")

## 6. 模型对比

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'RandomForest': test_metrics_rf,
    'XGBoost': test_metrics_xgb,
    'LightGBM': test_metrics_lgb,
    'MLP': test_metrics_mlp
})

print("\n模型对比:")
print(comparison.round(4))

## 7. 策略回测

In [ ]:
# 使用表现最好的模型进行回测
y_pred = model_xgb.predict(X_test)
test_features['pred_return'] = y_pred

backtester = Backtester(top_k=10)
result = backtester.run(test_features)

# 绘制结果
backtester.plot_results(result)

## 8. 一键运行完整流程

In [ ]:
from ml_framework.main import run_ml_pipeline

# 只需一行代码，更换模型类即可
result = run_ml_pipeline(XGBoostModel, {
    'n_estimators': 100,
    'max_depth': 5
})

# 切换到MLP模型
# result = run_ml_pipeline(MLPModel, {
#     'hidden_dims': [128, 64, 32],
#     'dropout_rate': 0.3
# })